In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
from collections import defaultdict

In [3]:
# ============================================================
# 修改为你的文件路径
# ============================================================
transcript_adata_path = '/home1/xyf/data/h5ad/ONT_final.h5ad'
gene_adata_path = '/home1/xyf/project/GLM7/get_data/data/NGS_all_feature.h5ad'

# 加载数据
print("加载数据...")
adata = sc.read_h5ad(transcript_adata_path)
adata_ngs = sc.read_h5ad(gene_adata_path)

print(f"✓ adata (转录本): {adata.n_obs:,} 细胞, {adata.n_vars:,} 转录本")
print(f"✓ adata_ngs (基因): {adata_ngs.n_obs:,} 细胞, {adata_ngs.n_vars:,} 基因")

加载数据...
✓ adata (转录本): 217,933 细胞, 113,690 转录本
✓ adata_ngs (基因): 213,961 细胞, 36,601 基因


In [4]:
adata.var

,Unnamed: 0,chrom,strand,length,exons,structural_category,associated_gene,associated_transcript,ref_length,ref_exons,...,dist_to_polyA_site,within_polyA_site,polyA_motif,polyA_dist,polyA_motif_found,ORF_seq,ratio_TSS,transcript_type,is_novel,category_merged
isoforms,,,,,,,,,,,,,,,,,,,,,
ENST00000000233,95199,chr7,+,1032,6,full-splice_match,ENSG00000004059,ENST00000000233,1032.0,6.0,...,NaN,NaN,NaN,NaN,NaN,MGLTVSALFSRIFGKKQMRILMVGLDAAGKTTILYKLKLGEIVTTI...,NaN,reference,False,full splice match
ENST00000000412,21832,chr12,-,2450,7,full-splice_match,ENSG00000003056,ENST00000000412,2450.0,7.0,...,NaN,NaN,NaN,NaN,NaN,MFPFYSCWRTGLLLLLLAVAVRESWQTEEKTCDLVGEKGKESEKEL...,NaN,reference,False,full splice match
ENST00000000442,15917,chr11,+,2274,7,full-splice_match,ENSG00000173153,ENST00000000442,2274.0,7.0,...,NaN,NaN,NaN,NaN,NaN,MSSQVVGIEPLYIKAEPASPDSPKGSSETETEPPVALAPGPAPTRC...,NaN,reference,False,full splice match
ENST00000001146,57371,chr2,-,4732,6,full-splice_match,ENSG00000003137,ENST00000001146,4732.0,6.0,...,NaN,NaN,NaN,NaN,NaN,MLFEGLDLVSALATLAACLVSVTLLLAVSQQLWQLRWAATRDKSCK...,NaN,reference,False,full splice match
ENST00000002125,57372,chr2,+,2184,10,full-splice_match,ENSG00000003509,ENST00000002125,2184.0,10.0,...,NaN,NaN,NaN,NaN,NaN,MSVLLRSGLGPLCAVARAAIPFIWRGKYFSSGNEPAENPVTPMLRH...,NaN,reference,False,full splice match
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
transcript982.chr11.nic,21831,chr11,-,1480,14,novel_in_catalog,ENSG00000177830,novel,1431.0,14.0,...,NaN,NaN,NaN,NaN,NaN,MARSWLTATSATQSQFSDKPVQDRGLVVTDLKAESVVLEHRSYCSA...,NaN,novel,True,novel in catalog
transcript982.chr4.nnic,83707,chr4,+,2847,17,novel_not_in_catalog,ENSG00000013810,novel,2799.0,16.0,...,NaN,NaN,NaN,NaN,NaN,MTLSPQEEVAAGQMASSSRSGPVKLEFDVSDGATSKRAPPPRRLGE...,NaN,novel,True,novel not in catalog
transcript9841.chr3.nic,78786,chr3,-,1610,5,full-splice_match,ENSG00000172037,ENST00000498377,1021.0,5.0,...,NaN,NaN,NaN,NaN,NaN,MVATRVLELSIPASAEQIQHLAGAIAERVRSLADVDAILARTVGDV...,NaN,reference,False,full splice match


In [5]:
transcript_to_gene = adata.var['associated_gene'].to_dict()

In [6]:
genes_in_adata_ngs = set(adata_ngs.var['gene_ids'])
genes_in_adata = set(g for g in transcript_to_gene.values() if pd.notna(g))

In [7]:
# 查看所有可用的细胞类型
print("可用的 developmental_system:")
print(adata_ngs.obs['developmental_system'].value_counts())

可用的 developmental_system:
developmental_system
somite               45216
neural progenitor    37601
limb                 22935
craniofacial         22435
splanchnic LPM       22063
head mesoderm        14551
somatic LPM          11655
neuron                7271
blood                 6069
schwann               5423
endothelium           5217
epidermis             4650
endoderm              4174
sensory neuron        3688
IM                    1013
Name: count, dtype: int64


In [8]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from tqdm import tqdm
import time

def extract_top_transcripts_simple(adata, adata_ngs, cell_type='limb', output_file=None,
                                   min_total_counts=10, min_cell_percentage=0.05,
                                   min_top_proportion=0.7):
    """
    直接在 adata 中提取特定细胞类型中每个基因表达量最高的转录本
    （不考虑 adata_ngs 的基因限制），并要求该转录本在该基因中的占比不低于阈值。
    
    参数:
    -----
    adata : AnnData
        包含转录本表达数据，adata.var['associated_gene'] 记录转录本对应的基因
    adata_ngs : AnnData
        仅用于获取细胞类型注释（adata_ngs.obs['developmental_system']）
    cell_type : str, optional (default='limb')
        要提取的细胞类型
    output_file : str, optional
        输出文件路径
    min_total_counts : float, optional (default=10)
        最小总表达量阈值（基于 adata 中的转录本表达）
    min_cell_percentage : float, optional (default=0.05)
        最小表达细胞比例（基于 adata 中的转录本表达）
    min_top_proportion : float, optional (default=0.7)
        最高转录本在该基因的表达占比（总表达量占比）最低阈值。
    
    返回:
    -----
    results_df : pd.DataFrame
        包含 gene、top_transcript 等信息的 DataFrame
    """
    
    # 自动设置输出文件名
    if output_file is None:
        output_file = f'top_transcripts_{cell_type}.txt'
    
    print("=" * 70)
    print(f"提取 {cell_type} 细胞中每个基因表达量最高的转录本")
    print(f"（基于 adata 转录本数据，不受 adata_ngs 基因集限制）")
    print(f"过滤条件: 总表达量 ≥ {min_total_counts}, 表达细胞比例 ≥ {min_cell_percentage*100}%, 最高转录本占比 ≥ {min_top_proportion*100}%")
    print("=" * 70)
    
    # 1. 筛选目标细胞
    print(f"\n[1/4] 筛选 {cell_type} 细胞...")
    target_cells = adata_ngs.obs[adata_ngs.obs['developmental_system'] == cell_type].index
    print(f"      找到 {len(target_cells):,} 个 {cell_type} 细胞")
    
    if len(target_cells) == 0:
        print(f"\n⚠️  错误: 没有找到 '{cell_type}' 类型的细胞！")
        print(f"可用的类型: {list(adata_ngs.obs['developmental_system'].unique())}")
        return None
    
    # 2. 筛选 adata 中的共同细胞
    print(f"\n[2/4] 在 adata 中匹配细胞...")
    common_cells = adata.obs_names.intersection(target_cells)
    print(f"      共有 {len(common_cells):,} 个共同的 {cell_type} 细胞")
    
    if len(common_cells) == 0:
        print("\n⚠️  错误: 没有找到共同的细胞！")
        return None
    
    adata_target = adata[common_cells, :]
    total_cells = len(common_cells)
    
    # 3. 按基因分组转录本
    print(f"\n[3/4] 按基因分组转录本...")
    start_time = time.time()
    
    transcript_to_gene = adata.var['associated_gene'].to_dict()
    
    # 统计基因
    all_genes = set()
    gene_transcript_dict = defaultdict(list)
    
    for transcript, gene in tqdm(transcript_to_gene.items(), 
                                 desc="      分组转录本", 
                                 ncols=70,
                                 disable=len(transcript_to_gene) < 10000):
        if pd.notna(gene) and transcript in adata_target.var_names:
            all_genes.add(gene)
            gene_transcript_dict[gene].append(transcript)
    
    elapsed_time = time.time() - start_time
    print(f"      ✓ 完成！耗时: {elapsed_time:.1f} 秒")
    print(f"      在 adata 中找到 {len(all_genes):,} 个基因")
    print(f"      需要处理 {len(gene_transcript_dict):,} 个基因")
    
    # 4. 选择每个基因表达量最高的转录本
    print(f"\n[4/4] 选择每个基因表达量最高的转录本...")
    start_time = time.time()
    
    results = []
    filtered_genes = []
    
    for gene, transcripts in tqdm(gene_transcript_dict.items(), 
                                  desc="      处理基因", 
                                  ncols=70):
        transcript_stats = {}
        
        for transcript in transcripts:
            expr_data = adata_target[:, transcript].X
            
            if hasattr(expr_data, 'toarray'):
                expr_array = expr_data.toarray().flatten()
            else:
                expr_array = np.array(expr_data).flatten()
            
            total_count = np.sum(expr_array)
            n_expressing_cells = np.sum(expr_array > 0)
            
            transcript_stats[transcript] = {
                'total_counts': total_count,
                'n_cells': n_expressing_cells
            }
        
        if transcript_stats:
            # 找到表达量最高的转录本
            top_transcript = max(transcript_stats.items(), 
                               key=lambda x: x[1]['total_counts'])[0]
            
            top_stats = transcript_stats[top_transcript]
            total_counts = top_stats['total_counts']
            n_cells = top_stats['n_cells']
            cell_percentage = n_cells / total_cells
            gene_total_counts = sum(stat['total_counts'] for stat in transcript_stats.values())
            top_fraction = top_stats['total_counts'] / gene_total_counts if gene_total_counts > 0 else 0
            
            passes_filters = (
                total_counts >= min_total_counts and
                cell_percentage >= min_cell_percentage and
                top_fraction >= min_top_proportion
            )
            
            if passes_filters:
                results.append({
                    'gene': gene,
                    'top_transcript': top_transcript,
                    'total_counts': total_counts,
                    'n_expressing_cells': n_cells,
                    'cell_percentage': cell_percentage,
                    'n_transcripts': len(transcripts),
                    'top_fraction': top_fraction
                })
            else:
                reasons = []
                if total_counts < min_total_counts:
                    reasons.append('low_total_counts')
                if cell_percentage < min_cell_percentage:
                    reasons.append('low_cell_percentage')
                if top_fraction < min_top_proportion:
                    reasons.append('top_fraction_below_threshold')
                filtered_genes.append({
                    'gene': gene,
                    'top_transcript': top_transcript,
                    'total_counts': total_counts,
                    'n_expressing_cells': n_cells,
                    'cell_percentage': cell_percentage,
                    'top_fraction': top_fraction,
                    'reason': '|'.join(reasons) if reasons else 'filtered'
                })
    
    elapsed_time = time.time() - start_time
    print(f"      ✓ 完成！耗时: {elapsed_time:.1f} 秒")
    
    # 转换为 DataFrame 并排序
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df = results_df.sort_values('gene').reset_index(drop=True)
    
    # 统计互斥的过滤原因组合
    reason_labels = {
        'low_total_counts': '总表达量不足',
        'low_cell_percentage': '细胞比例不足',
        'top_fraction_below_threshold': '最高转录本占比不足',
        'unknown': '未标明原因'
    }
    reason_combo_counter = Counter()
    for info in filtered_genes:
        raw_reason = info.get('reason', '')
        combo = tuple(sorted(r for r in raw_reason.split('|') if r))
        if not combo:
            combo = ('unknown',)
        reason_combo_counter[combo] += 1
    total_filtered = len(filtered_genes)
    
    # 保存结果 - 只保存转录本名称
    print(f"\n保存结果...")
    with open(output_file, 'w') as f:
        for _, row in results_df.iterrows():
            f.write(f"{row['top_transcript']}\n")
    
    # 保存详细统计信息
    stats_file = output_file.replace('.txt', '_stats.txt')
    with open(stats_file, 'w') as f:
        f.write(f"分析统计信息\n")
        f.write(f"{'='*70}\n")
        f.write(f"细胞类型: {cell_type}\n")
        f.write(f"细胞数量: {total_cells}\n")
        f.write(f"数据来源: adata (转录本数据)\n\n")
        
        f.write(f"基因统计:\n")
        f.write(f"  adata 中的基因总数: {len(all_genes)}\n")
        f.write(f"  候选基因: {len(gene_transcript_dict)}\n")
        f.write(f"  通过过滤: {len(results_df)}\n")
        f.write(f"  被过滤: {len(filtered_genes)}\n")
        f.write(f"  过滤率: {len(filtered_genes)/len(gene_transcript_dict)*100:.2f}%\n\n")
        
        if total_filtered > 0:
            f.write(f"过滤原因互斥分布:\n")
            for combo, count in reason_combo_counter.most_common():
                combo_label = ' + '.join(reason_labels.get(r, r) for r in combo)
                f.write(f"  - {combo_label}: {count} ({count/total_filtered*100:.1f}%)\n")
            f.write("\n")
        
        f.write(f"过滤条件:\n")
        f.write(f"  最小总表达量: {min_total_counts}\n")
        f.write(f"  最小细胞比例: {min_cell_percentage*100}%\n")
        f.write(f"  最低占比阈值: {min_top_proportion*100}%\n")
        
        if len(results_df) > 0:
            f.write(f"\n转录本统计:\n")
            f.write(f"  平均每个基因的转录本数: {results_df['n_transcripts'].mean():.1f}\n")
            f.write(f"  最多转录本数: {results_df['n_transcripts'].max()}\n")
        
        if filtered_genes:
            f.write(f"\n被过滤基因示例（前20个）：\n")
            for i, info in enumerate(filtered_genes[:20]):
                f.write(
                    f"  {info['gene']}: counts={info['total_counts']:.0f}, "
                    f"cells={info['n_expressing_cells']} ({info['cell_percentage']*100:.1f}%), "
                    f"top_fraction={info['top_fraction']*100:.1f}%, reason={info['reason']}\n"
                )
    
    # 保存完整的详细结果
    if len(results_df) > 0:
        detailed_file = output_file.replace('.txt', '_detailed.tsv')
        results_df.to_csv(detailed_file, sep='\t', index=False)
    
    print(f"\n{'='*70}")
    print(f"✓ 完成！")
    print(f"✓ adata 中的基因: {len(all_genes):,} 个")
    print(f"✓ 候选基因: {len(gene_transcript_dict):,} 个")
    print(f"✓ 通过过滤: {len(results_df):,} 个 ({len(results_df)/len(gene_transcript_dict)*100:.1f}%)")
    print(f"✓ 被过滤: {len(filtered_genes):,} 个 ({len(filtered_genes)/len(gene_transcript_dict)*100:.1f}%)")
    
    if total_filtered > 0:
        print(f"\n过滤原因互斥分布:")
        for combo, count in reason_combo_counter.most_common():
            combo_label = ' + '.join(reason_labels.get(r, r) for r in combo)
            print(f"  - {combo_label}: {count:,} 个 ({count/total_filtered*100:.1f}%)")
    
    if len(results_df) > 0:
        print(f"\n输出文件:")
        print(f"  - 转录本列表: {output_file}")
        print(f"  - 统计信息: {stats_file}")
        print(f"  - 详细结果: {detailed_file}")
    print(f"{'='*70}")
    
    return results_df


In [9]:
adata

AnnData object with n_obs × n_vars = 217933 × 113690
    var: 'Unnamed: 0', 'chrom', 'strand', 'length', 'exons', 'structural_category', 'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons', 'diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS', 'subcategory', 'RTS_stage', 'all_canonical', 'min_sample_cov', 'min_cov', 'min_cov_pos', 'sd_cov', 'FL', 'n_indels', 'n_indels_junc', 'bite', 'iso_exp', 'gene_exp', 'ratio_exp', 'FSM_class', 'coding', 'ORF_length', 'CDS_length', 'CDS_start', 'CDS_end', 'CDS_genomic_start', 'CDS_genomic_end', 'predicted_NMD', 'perc_A_downstream_TTS', 'seq_A_downstream_TTS', 'dist_to_CAGE_peak', 'within_CAGE_peak', 'dist_to_polyA_site', 'within_polyA_site', 'polyA_motif', 'polyA_dist', 'polyA_motif_found', 'ORF_seq', 'ratio_TSS', 'transcript_type', 'is_novel', 'category_merged'

In [10]:
adata_ngs

AnnData object with n_obs × n_vars = 213961 × 36601
    obs: 'celltype', 'total_UMIs_NGS', 'developmental_system', 'sample', 'total_UMIs_ONT', 'embryos_id', 'stage', 'part', 'embryos_id_emb2', 'embryos_id_emb3', 'embryos_id_emb4', 'embryos_id_emb5', 'embryos_id_emb6', 'embryos_id_emb7', 'embryos_id_emb8', 'embryos_id_emb9', 'stage_CS12', 'stage_CS13', 'stage_CS14', 'stage_CS15', 'developmental_system_blood', 'developmental_system_craniofacial', 'developmental_system_endoderm', 'developmental_system_endothelium', 'developmental_system_epidermis', 'developmental_system_head mesoderm', 'developmental_system_limb', 'developmental_system_neural progenitor', 'developmental_system_neuron', 'developmental_system_schwann', 'developmental_system_sensory neuron', 'developmental_system_somatic LPM', 'developmental_system_somite', 'developmental_system_splanchnic LPM', 'log1p_total_UMIs_NGS'
    var: 'gene_ids', 'feature_type', 'ensembl_ids', 'highly_variable', 'means', 'dispersions', 'dispersions_

In [11]:
with open('/home1/xyf/data/openspliceai_data/gtf/all_isoform.txt', 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

In [12]:
adata = adata[:,lines].copy()

In [13]:
adata

AnnData object with n_obs × n_vars = 217933 × 64137
    var: 'Unnamed: 0', 'chrom', 'strand', 'length', 'exons', 'structural_category', 'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons', 'diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS', 'subcategory', 'RTS_stage', 'all_canonical', 'min_sample_cov', 'min_cov', 'min_cov_pos', 'sd_cov', 'FL', 'n_indels', 'n_indels_junc', 'bite', 'iso_exp', 'gene_exp', 'ratio_exp', 'FSM_class', 'coding', 'ORF_length', 'CDS_length', 'CDS_start', 'CDS_end', 'CDS_genomic_start', 'CDS_genomic_end', 'predicted_NMD', 'perc_A_downstream_TTS', 'seq_A_downstream_TTS', 'dist_to_CAGE_peak', 'within_CAGE_peak', 'dist_to_polyA_site', 'within_polyA_site', 'polyA_motif', 'polyA_dist', 'polyA_motif_found', 'ORF_seq', 'ratio_TSS', 'transcript_type', 'is_novel', 'category_merged'

In [14]:
results = extract_top_transcripts_simple(
    adata=adata,
    adata_ngs=adata_ngs,
    cell_type='IM',
    min_total_counts=2,        # 不过滤表达量
    min_cell_percentage=0.01,     # 不过滤细胞比例
    min_top_proportion=0.7,
    output_file='/home1/xyf/data/openspliceai_tissue_data/isoform/IM2.txt'
)

提取 IM 细胞中每个基因表达量最高的转录本
（基于 adata 转录本数据，不受 adata_ngs 基因集限制）
过滤条件: 总表达量 ≥ 2, 表达细胞比例 ≥ 1.0%, 最高转录本占比 ≥ 70.0%

[1/4] 筛选 IM 细胞...
      找到 1,013 个 IM 细胞

[2/4] 在 adata 中匹配细胞...
      共有 1,013 个共同的 IM 细胞

[3/4] 按基因分组转录本...


      分组转录本:   0%|                     | 0/64137 [00:00<?, ?it/s]

      分组转录本: 100%|█████| 64137/64137 [00:00<00:00, 603130.50it/s]


      ✓ 完成！耗时: 0.1 秒
      在 adata 中找到 22,232 个基因
      需要处理 22,232 个基因

[4/4] 选择每个基因表达量最高的转录本...


      处理基因: 100%|██████████| 22232/22232 [03:24<00:00, 108.49it/s]


      ✓ 完成！耗时: 204.9 秒

保存结果...

✓ 完成！
✓ adata 中的基因: 22,232 个
✓ 候选基因: 22,232 个
✓ 通过过滤: 4,009 个 (18.0%)
✓ 被过滤: 18,223 个 (82.0%)

过滤原因互斥分布:
  - 细胞比例不足 + 总表达量不足 + 最高转录本占比不足: 7,312 个 (40.1%)
  - 细胞比例不足: 3,158 个 (17.3%)
  - 细胞比例不足 + 最高转录本占比不足: 2,998 个 (16.5%)
  - 最高转录本占比不足: 2,793 个 (15.3%)
  - 细胞比例不足 + 总表达量不足: 1,962 个 (10.8%)

输出文件:
  - 转录本列表: /home1/xyf/data/openspliceai_tissue_data/isoform/IM2.txt
  - 统计信息: /home1/xyf/data/openspliceai_tissue_data/isoform/IM2_stats.txt
  - 详细结果: /home1/xyf/data/openspliceai_tissue_data/isoform/IM2_detailed.tsv


In [14]:
from pathlib import Path

# 批量计算多个组织的最高表达转录本，输出到指定目录
tissue_targets = [
    'somite',
    'neural progenitor',
    'limb',
    'craniofacial',
    'splanchnic LPM',
    'head mesoderm',
    'somatic LPM',
    'neuron',
    'blood',
    'schwann',
    'endothelium',
    'epidermis',
    'endoderm',
    'sensory neuron',
    'IM',
]

output_dir = Path('/home1/xyf/data/openspliceai_tissue_data/isoform')
output_dir.mkdir(parents=True, exist_ok=True)

run_params = dict(
    min_total_counts=10,
    min_cell_percentage=0.01,
    min_top_proportion=0.7,
)

batch_results = {}

for tissue in tissue_targets:
    safe_name = tissue.replace(' ', '_').replace('/', '_')
    output_file = output_dir / f"{safe_name}.txt"
    print(f"\n>>> 处理 {tissue}, 输出: {output_file}")
    try:
        batch_results[tissue] = extract_top_transcripts_simple(
            adata=adata,
            adata_ngs=adata_ngs,
            cell_type=tissue,
            output_file=str(output_file),
            **run_params,
        )
    except Exception as exc:
        batch_results[tissue] = None
        print(f"处理 {tissue} 失败: {exc}")



>>> 处理 somite, 输出: /home1/xyf/data/openspliceai_tissue_data/isoform/somite.txt
提取 somite 细胞中每个基因表达量最高的转录本
（基于 adata 转录本数据，不受 adata_ngs 基因集限制）
过滤条件: 总表达量 ≥ 10, 表达细胞比例 ≥ 1.0%, 最高转录本占比 ≥ 70.0%

[1/4] 筛选 somite 细胞...
      找到 45,216 个 somite 细胞

[2/4] 在 adata 中匹配细胞...
      共有 45,216 个共同的 somite 细胞

[3/4] 按基因分组转录本...


      分组转录本:   0%|                     | 0/64137 [00:00<?, ?it/s]

      分组转录本: 100%|█████| 64137/64137 [00:00<00:00, 626427.92it/s]


      ✓ 完成！耗时: 0.1 秒
      在 adata 中找到 22,232 个基因
      需要处理 22,232 个基因

[4/4] 选择每个基因表达量最高的转录本...


      处理基因: 100%|███████████| 22232/22232 [05:16<00:00, 70.26it/s]


...

      ✓ 完成！耗时: 213.9 秒

保存结果...

✓ 完成！
✓ adata 中的基因: 22,232 个
✓ 候选基因: 22,232 个
✓ 通过过滤: 3,605 个 (16.2%)
✓ 被过滤: 18,627 个 (83.8%)

过滤原因互斥分布:
  - 细胞比例不足 + 总表达量不足 + 最高转录本占比不足: 7,147 个 (38.4%)
  - 细胞比例不足 + 总表达量不足: 4,627 个 (24.8%)
  - 最高转录本占比不足: 2,439 个 (13.1%)
  - 细胞比例不足 + 最高转录本占比不足: 2,436 个 (13.1%)
  - 细胞比例不足: 1,978 个 (10.6%)

输出文件:
  - 转录本列表: /home1/xyf/data/openspliceai_tissue_data/isoform/sensory_neuron.txt
  - 统计信息: /home1/xyf/data/openspliceai_tissue_data/isoform/sensory_neuron_stats.txt
  - 详细结果: /home1/xyf/data/openspliceai_tissue_data/isoform/sensory_neuron_detailed.tsv

>>> 处理 IM, 输出: /home1/xyf/data/openspliceai_tissue_data/isoform/IM.txt
提取 IM 细胞中每个基因表达量最高的转录本
（基于 adata 转录本数据，不受 adata_ngs 基因集限制）
过滤条件: 总表达量 ≥ 10, 表达细胞比例 ≥ 1.0%, 最高转录本占比 ≥ 70.0%

[1/4] 筛选 IM 细胞...
      找到 1,013 个 IM 细胞

[2/4] 在 adata 中匹配细胞...
      共有 1,013 个共同的 IM 细胞

[3/4] 按基因分组转录本...


      分组转录本: 100%|█████| 64137/64137 [00:00<00:00, 609320.39it/s]


      ✓ 完成！耗时: 0.1 秒
      在 adata 中找到 22,232 个基因
      需要处理 22,232 个基因

[4/4] 选择每个基因表达量最高的转录本...


      处理基因: 100%|██████████| 22232/22232 [03:25<00:00, 108.44it/s]


      ✓ 完成！耗时: 205.0 秒

保存结果...

✓ 完成！
✓ adata 中的基因: 22,232 个
✓ 候选基因: 22,232 个
✓ 通过过滤: 4,009 个 (18.0%)
✓ 被过滤: 18,223 个 (82.0%)

过滤原因互斥分布:
  - 细胞比例不足 + 总表达量不足 + 最高转录本占比不足: 10,046 个 (55.1%)
  - 细胞比例不足 + 总表达量不足: 4,861 个 (26.7%)
  - 最高转录本占比不足: 2,793 个 (15.3%)
  - 细胞比例不足 + 最高转录本占比不足: 264 个 (1.4%)
  - 细胞比例不足: 259 个 (1.4%)

输出文件:
  - 转录本列表: /home1/xyf/data/openspliceai_tissue_data/isoform/IM.txt
  - 统计信息: /home1/xyf/data/openspliceai_tissue_data/isoform/IM_stats.txt
  - 详细结果: /home1/xyf/data/openspliceai_tissue_data/isoform/IM_detailed.tsv
